# TransformerLens to TorchLens Migration

This 50-minute-track notebook is intentionally self-contained: it uses a seeded tiny PyTorch model, no downloads, and only public TorchLens APIs. The goal is to show the full workflow shape you would use on a larger model while keeping every cell runnable on CPU.


TransformerLens users often start from `run_with_cache`, named hook points, and activation patching. TorchLens uses regular PyTorch execution and records all operations, so the migration is mostly about replacing cache keys with selector queries and replacing hook-point names with resolved TorchLens sites.


In [ ]:
from __future__ import annotations

import math
from typing import Any

import matplotlib.pyplot as plt
import torch
from torch import nn

import torchlens as tl

SEED = 1603
torch.manual_seed(SEED)
torch.set_grad_enabled(False)

try:
    import transformer_lens as transformer_lens  # type: ignore[import-not-found]
except ImportError:
    transformer_lens = None


class MockHookedTransformer(nn.Module):
    """Tiny local stand-in for a TransformerLens HookedTransformer."""

    def __init__(self, vocab_size: int = 12, width: int = 10) -> None:
        """Initialize a minimal token model."""
        super().__init__()
        self.embed = nn.Embedding(vocab_size, width)
        self.blocks = nn.ModuleList([nn.Sequential(nn.Linear(width, width), nn.ReLU())])
        self.unembed = nn.Linear(width, vocab_size, bias=False)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        """Return token logits."""
        x = self.embed(tokens)
        for block in self.blocks:
            x = x + block(x)
        return self.unembed(x)

In [ ]:
if transformer_lens is None:
    print("TransformerLens is not installed; using a local mock with the same workflow shape.")
else:
    print(
        "TransformerLens is installed; replace MockHookedTransformer with a real HookedTransformer."
    )

model = MockHookedTransformer().eval()
tokens = torch.tensor([[1, 4, 5, 2]])
log = tl.log_forward_pass(model, tokens, vis_opt="none", intervention_ready=True)
print(f"TorchLens saved {len(log.layer_list)} operation-level layers")
print([site.layer_label for site in log.find_sites(tl.func("relu"))])

Migration pattern: `cache['blocks.0.mlp.hook_post']` becomes a selector query such as `tl.func('relu') & tl.in_module('blocks.0')`, then `find_sites(...).first().activation` returns the tensor.


In [ ]:
mlp_post = log.find_sites(tl.func("relu") & tl.in_module("blocks.0")).first()
print(mlp_post.layer_label)
print(tuple(mlp_post.activation.shape))

For intervention-style code, `run_with_hooks` maps to a fork plus sticky hooks. Replay is the fastest option when the logged graph is stable; rerun is the closer analogue when the original model should execute again.


In [ ]:
ablated = log.fork("zero_mlp_post")
ablated.attach_hooks(tl.label(mlp_post.layer_label), tl.zero_ablate())
ablated.replay()

clean_out = log.layer_list[-1].activation
ablated_out = ablated.layer_list[-1].activation
print(f"output L2 change: {torch.linalg.vector_norm(ablated_out - clean_out):.4f}")
print(f"last engine: {ablated.last_run_records()[-1].engine}")